# ACOPF settlement — does the money close?

The transient-stability report (`KKT_Settlement.ipynb`) prices demand at
$\pi_k = \lambda_k + \text{stability}_k$ and generation at
$\hat\pi_g = \lambda_g + \text{charge}_g$, and its residual is **not** expected to
vanish — that is the finding. This notebook is the baseline that claim is read
against: the same accounting on a plain ACOPF, where the residual **is** an exact
algebraic identity. If it does not close here, the problem is the export or the
solver tolerance, not economics.

Everything below comes off disk. Nothing is re-solved and no sensitivity is
recomputed.

## Two identities, and they are not the same statement

Which dual belongs to which identity is the whole point, so they are kept apart.

### 1. Network identity — where the operator's money comes from

The active-power balance row (`functions_4_eqconst.jl:157`) is

$$P_{g,k} \;-\; P_{d,k} \;-\; g_{sh,k}V_k^2 \;-\; \textstyle\sum_{\text{flows}} \;=\; 0,$$

with $\lambda_k$ its dual. The primal LMP is $+\lambda$, not $-\lambda$. Multiply by
$\lambda_k$, sum over buses, and the merchandising surplus is forced:

$$\mathrm{MS}_P \;\equiv\; \sum_k \lambda_k\,(P_{d,k}-P_{g,k})
  \;=\; -\sum_k \lambda_k P_{sh,k} \;-\; \sum_{\text{branches}} \big(\lambda_i P_{ik} + \lambda_k P_{ki}\big).$$

Each branch term splits exactly, and in two different ways:

$$\lambda_i P_{ik} + \lambda_k P_{ki}
 \;=\; \underbrace{\tfrac12(\lambda_i{+}\lambda_k)(P_{ik}{+}P_{ki})}_{\text{mean price} \times \text{loss}}
     + \underbrace{\tfrac12(\lambda_i{-}\lambda_k)(P_{ik}{-}P_{ki})}_{\text{price spread} \times \text{mean directed flow}}
 \;=\; \underbrace{\lambda_k (P_{ik}{+}P_{ki})}_{\text{to-side price} \times \text{loss}}
     + \underbrace{(\lambda_i{-}\lambda_k) P_{ik}}_{\text{textbook congestion rent}} .$$

The first split is orientation-independent — it does not care which end you called
"from". The second matches textbook language but flips sign if the branch is
entered the other way round. Both are reported; the first is the one asserted.

**$\mu_V$, $\mu_\theta$ and $\mu_{P_g}$ do not appear here.** No voltage limit and
no generator bound contributes a euro to the operator's surplus. The identical
identity holds for reactive power with $\mu^Q$.

### 2. Stationarity identity — where the generator's money comes from

$\partial\mathcal{L}/\partial P_g = 0$, with the bound rows written
$\underline{P}-P_g \le 0$ and $P_g-\overline{P}\le 0$
(`bound_builders.jl:160,169`), gives

$$\lambda_{\mathrm{bus}(g)} \;=\; C'_g(P_g) \;+\; \mu^{P_g}_{\text{lower}} \;-\; \mu^{P_g}_{\text{upper}} ,$$

so generator profit $(\lambda - C')\,P_g$ **is** the scarcity rent on whichever
bound binds. This is where the $\mu$'s live. A unit held at $P_{\min}$ under a
linear cost has $\lambda < C'$ and loses money — that is a make-whole, and it is
reported as one.

### 3. The demand pin

With `pd_as_variable = true` the run also carries $\sigma_k$, the dual of
$P_d[k] - p_d^0[k] = 0$, which is $\partial\mathcal{L}/\partial P_{d}(k)$ by
construction. On a dispatch-only ACOPF it must equal $\lambda_k$. Proving that
here is what licenses reading $\sigma - \lambda$ as stability content once the
transient rows are added.

## What is asserted

| # | check | tolerance |
|---|---|---|
| 1 | `buses_report` satisfies its own balance row | 1e-6 MW |
| 2 | $\lambda_k = \sigma_k$ | 1e-6 EUR/MW |
| 3 | $\mathrm{MS}_P$ closes on the branch decomposition | 1e-6 EUR/h |
| 4 | $\mathrm{MS}_Q$ closes on the branch decomposition | 1e-6 EUR/h |
| 5 | both branch splits reproduce the same total | 1e-9 EUR/h |
| 6 | `kkt_Pg_terms.csv` `RESIDUAL` is zero, and its $\lambda$ matches | 1e-8 |

A failed assertion fails the `nbconvert --execute` render. That is the test.


In [ ]:
import os, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)

TOL = 1e-6   # EUR/h. Above this it is a modelling error, not rounding.

# --- locate the run -------------------------------------------------------------------
# ACOPF_RUN_DIR may name a run folder directly, or a RESULTS root, in which case the
# newest suitable run wins. Same resolution as the KKT notebooks.
DUALS = 'Dispatch/CSV_duals'
env   = os.environ.get('ACOPF_RUN_DIR', '')
cand  = pathlib.Path(env) if env else pathlib.Path.cwd()

def is_run(p):
    return (p / 'run_manifest.toml').is_file() and (p / DUALS / 'dual_P_balance.csv').is_file()

if is_run(cand):
    RUN = cand
else:
    runs = sorted(p for p in cand.glob('Results - *') if is_run(p))
    if not runs:
        raise SystemExit(f'No run with {DUALS}/dual_P_balance.csv found under {cand}. '
                         'Set ACOPF_RUN_DIR to a run folder or a RESULTS root.')
    RUN = runs[-1]

rd = lambda *p: pd.read_csv(RUN.joinpath(*p), sep=';')

def rd_opt(*p):
    # A file that may legitimately be absent: which dual families exist depends on the
    # DispatchConfig flags of the run (DispatchDualRegistry.jl only writes what was built).
    path = RUN.joinpath(*p)
    return pd.read_csv(path, sep=';') if path.is_file() else None

# --- manifest: the run describes itself -------------------------------------------------
# Flat parse, first occurrence wins - identical to the sibling notebooks. Enough for the
# scalars and the c0/c1/c2 arrays; nothing here needs the section nesting.
man = {}
for line in (RUN / 'run_manifest.toml').read_text(encoding='utf8').splitlines():
    s = line.strip()
    if '=' in s and not s.startswith('['):
        k, _, v = s.partition('=')
        man.setdefault(k.strip(), v.strip().strip('"'))

def arr(key):
    raw  = man.get(key, '')
    body = raw.strip('[]').strip()
    return [float(x) for x in body.split(',')] if body else []

CASE       = man.get('case', '?')
SBASE      = float(man.get('base_MVA', 100.0))
COST_TYPE  = man.get('cost_type', 'linear')
OBJ_MVA    = float(man.get('objective_MVA', 'nan'))
TRANS_STAB = man.get('trans_stab', 'false') == 'true'
SBRANCH    = man.get('ineq_sbranch_upper', 'false') == 'true'
PD_PIN     = man.get('pd_as_variable', 'false') == 'true'
SG_UPPER   = man.get('ineq_sg_upper', 'false') == 'true'

if TRANS_STAB:
    raise SystemExit(f'{RUN.name} is a transient-stability run (trans_stab = true). This '
                     'notebook is the ACOPF baseline; use KKT_Settlement.ipynb for that run.')

print(f'run        {RUN.name}')
print(f'case       {CASE}    base_MVA = {SBASE:g}    cost_type = {COST_TYPE}')
print(f'objective  {OBJ_MVA:.6f} EUR/h   ({man.get("termination_status", "?")})')
print(f'flags      ineq_sbranch_upper={SBRANCH}   pd_as_variable={PD_PIN}   ineq_sg_upper={SG_UPPER}')
if not SBRANCH:
    print('\nNOTE: no branch thermal limit was built, so congestion rent is identically zero')
    print('      and the surplus below is pure loss rent. That is a valid closure test, but')
    print('      it exercises nothing. Set ineq_sbranch_upper = true to price a binding line.')


In [ ]:
# --- primal tables ----------------------------------------------------------------------
BUS  = rd('Dispatch', 'CSV', 'buses_report.csv').set_index('BUS').sort_index()
CIRC = rd('Dispatch', 'CSV', 'circuits_report.csv')
GENR = rd('Dispatch', 'CSV', 'generators_report.csv')

buses    = BUS.index
branches = CIRC['ID_CIRC'].values
gen_ids  = GENR['ID'].values

# --- duals ------------------------------------------------------------------------------
# A missing OPTIONAL family becomes zeros plus one printed line; a missing REQUIRED one is
# fatal, because there is no price without it.
def dual_series(fname, col, index_col, index, required=False):
    df = rd_opt('Dispatch', 'CSV_duals', fname)
    if df is None:
        if required:
            raise SystemExit(f'{fname} is missing from {RUN.name}/{DUALS}. '
                             'The run needs save_duals = true.')
        print(f'  absent, read as 0 : {fname}')
        return pd.Series(0.0, index=index, name=col)
    return df.set_index(index_col)[col].reindex(index).fillna(0.0).rename(col)

lam   = dual_series('dual_P_balance.csv', 'Dual_P_Balance', 'Bus_ID',    buses,    required=True)
muQ   = dual_series('dual_Q_balance.csv', 'Dual_Q_Balance', 'Bus_ID',    buses,    required=True)
sigma = dual_series('dual_Pd_pin.csv',    'Dual_Pd_Pin',    'Bus_ID',    buses)
v_lo  = dual_series('dual_V_lower.csv',   'Dual_V_Lower',   'Bus_ID',    buses)
v_up  = dual_series('dual_V_upper.csv',   'Dual_V_Upper',   'Bus_ID',    buses)
th_lo = dual_series('dual_\u03b8_lower.csv', 'Dual_\u03b8_Lower', 'Bus_ID', buses)
th_up = dual_series('dual_\u03b8_upper.csv', 'Dual_\u03b8_Upper', 'Bus_ID', buses)
pg_lo = dual_series('dual_Pg_lower.csv',  'Dual_Pg_Lower',  'Gen_ID',    gen_ids)
pg_up = dual_series('dual_Pg_upper.csv',  'Dual_Pg_Upper',  'Gen_ID',    gen_ids)
qg_lo = dual_series('dual_Qg_lower.csv',  'Dual_Qg_Lower',  'Gen_ID',    gen_ids)
qg_up = dual_series('dual_Qg_upper.csv',  'Dual_Qg_Upper',  'Gen_ID',    gen_ids)
sg_up = dual_series('dual_Sg_Upper.csv',  'Dual_Sg_Upper',  'Gen_ID',    gen_ids)
s_ik  = dual_series('dual_Sik_Upper.csv', 'Dual_Sik_Upper', 'Branch_ID', branches)
s_ki  = dual_series('dual_Ski_Upper.csv', 'Dual_Ski_Upper', 'Branch_ID', branches)

print(f'\n{len(buses)} buses, {len(branches)} branches, {len(gen_ids)} generators')
print(f'lambda range  {lam.min():10.6f} .. {lam.max():10.6f} EUR/MW   spread {lam.max()-lam.min():.6f}')
print(f'mu_Q   range  {muQ.min():10.6f} .. {muQ.max():10.6f} EUR/MVAr')


In [ ]:
# --- check 1: does the exported bus table satisfy its own balance row? -------------------
# P balance (functions_4_eqconst.jl:157):  P_g - p_d - g_sh*V^2 - sum(flows) = 0,
# and buses_report writes the flow sum as P_MW. So PG - PD - Psh must equal P_MW.
res_P = BUS.PG_MW - BUS.PD_MW - BUS.Psh_MW - BUS.P_MW

# The Q row carries the shunt with the OPPOSITE sign (+ b_sh*V^2, line 225), and the report
# may write Qsh with either convention. Detect it rather than guess; on a case with no bus
# susceptance both forms are identical and the choice is harmless.
res_Q_minus = BUS.QG_MVAr - BUS.QD_MVAr - BUS.Qsh_MVAr - BUS.Q_MVAr
res_Q_plus  = BUS.QG_MVAr - BUS.QD_MVAr + BUS.Qsh_MVAr - BUS.Q_MVAr
QSH_SIGN = -1.0 if res_Q_minus.abs().max() <= res_Q_plus.abs().max() else +1.0
res_Q    = res_Q_minus if QSH_SIGN < 0 else res_Q_plus
Qsh_eff  = QSH_SIGN * BUS.Qsh_MVAr          # signed so that QG - QD - Qsh_eff = Q_MVAr

print(f'max |P balance residual|   {res_P.abs().max():.3e} MW')
print(f'max |Q balance residual|   {res_Q.abs().max():.3e} MVAr   (Qsh sign {QSH_SIGN:+.0f})')
assert res_P.abs().max() < 1e-6, 'buses_report P columns do not satisfy the balance row'
assert res_Q.abs().max() < 1e-6, 'buses_report Q columns do not satisfy the balance row'

# --- check 2: the demand pin must reproduce the balance dual -----------------------------
HAS_PIN   = PD_PIN and (RUN / DUALS / 'dual_Pd_pin.csv').is_file()
SIGMA_GAP = float((lam - sigma).abs().max()) if HAS_PIN else float('nan')
if HAS_PIN:
    print(f'max |lambda_k - sigma_k|   {SIGMA_GAP:.3e} EUR/MW')
    assert SIGMA_GAP < TOL, (
        'the demand-pin dual disagrees with the balance dual on a dispatch-only ACOPF. '
        'sigma_k is dL/dP_d(k); with no transient rows carrying the demand it can only be '
        'lambda_k. A gap here means the pin is coupled to something else in this model.')
    print('  -> sigma == lambda: the demand pin is the balance dual, as it must be.')
else:
    print('no dual_Pd_pin.csv: run with pd_as_variable = true to get the sigma cross-check.')


In [ ]:
# --- Tier 1: the aggregate settlement ----------------------------------------------------
# Everything nodal, everything at the balance dual of its own bus.
pay_load_P = float((lam * BUS.PD_MW).sum())
pay_gen_P  = float((lam * BUS.PG_MW).sum())
MS_P       = pay_load_P - pay_gen_P

pay_load_Q = float((muQ * BUS.QD_MVAr).sum())
pay_gen_Q  = float((muQ * BUS.QG_MVAr).sum())
MS_Q       = pay_load_Q - pay_gen_Q

T1 = pd.DataFrame({
    'quantity': ['load pays', 'generation paid', 'operator surplus (load - gen)'],
    'active [EUR/h]':   [pay_load_P, pay_gen_P, MS_P],
    'reactive [EUR/h]': [pay_load_Q, pay_gen_Q, MS_Q],
})
T1['total [EUR/h]'] = T1['active [EUR/h]'] + T1['reactive [EUR/h]']
print('TIER 1 - aggregate settlement\n')
print(T1.to_string(index=False, float_format=lambda x: f'{x:14.6f}'))
print(f'\ntotal demand {BUS.PD_MW.sum():.4f} MW, total generation {BUS.PG_MW.sum():.4f} MW, '
      f'losses {BUS.PG_MW.sum() - BUS.PD_MW.sum() - BUS.Psh_MW.sum():.4f} MW')
print('\nA negative surplus means the operator tops the settlement up rather than collecting.')
print('Its sign is not decided in advance: losses push it down, the loss-driven price')
print('spread pushes it up. Which wins is what Tier 2 decomposes.')


In [ ]:
# --- Tier 2: the surplus, rebuilt on the branches -----------------------------------------
# MS_P = -sum_k lambda_k*P_sh,k - sum_br (lambda_i*P_ik + lambda_k*P_ki). Signs are chosen so
# that every "rent" column ADDS UP TO the surplus rather than to its negative.
B = CIRC.rename(columns={'ID_CIRC': 'branch', 'FROM_BUS': 'i', 'TO_BUS': 'k'}).copy()

li = B['i'].map(lam).to_numpy(float)
lk = B['k'].map(lam).to_numpy(float)
mi = B['i'].map(muQ).to_numpy(float)
mk = B['k'].map(muQ).to_numpy(float)
Pik, Pki = B.Pik_MW.to_numpy(float),   B.Pki_MW.to_numpy(float)
Qik, Qki = B.Qik_MVAr.to_numpy(float), B.Qki_MVAr.to_numpy(float)

B['loss_P']   = Pik + Pki                                   # = Ploss_MW, by definition
B['rent_P']   = -(li * Pik + lk * Pki)                       # this branch's share of MS_P
B['loss_rent_P']   = -0.5 * (li + lk) * (Pik + Pki)          # symmetric split ...
B['spread_rent_P'] = -0.5 * (li - lk) * (Pik - Pki)          # ... orientation-independent
B['loss_rent_dir_P']   = -lk * (Pik + Pki)                   # directed split ...
B['spread_rent_dir_P'] = (lk - li) * Pik                     # ... textbook congestion rent

B['loss_Q']   = Qik + Qki
B['rent_Q']   = -(mi * Qik + mk * Qki)
B['loss_rent_Q']   = -0.5 * (mi + mk) * (Qik + Qki)
B['spread_rent_Q'] = -0.5 * (mi - mk) * (Qik - Qki)

B['mu_Sik'] = B.branch.map(s_ik)
B['mu_Ski'] = B.branch.map(s_ki)
B['binding'] = (B.mu_Sik.abs() > 1e-6) | (B.mu_Ski.abs() > 1e-6)

shunt_rent_P = float(-(lam * BUS.Psh_MW).sum())
shunt_rent_Q = float(-(muQ * Qsh_eff).sum())

# --- check 5: the two splits are the same money ------------------------------------------
assert (B.loss_rent_P + B.spread_rent_P - B.rent_P).abs().max() < 1e-9
assert (B.loss_rent_dir_P + B.spread_rent_dir_P - B.rent_P).abs().max() < 1e-9
assert (B.loss_rent_Q + B.spread_rent_Q - B.rent_Q).abs().max() < 1e-9

# --- checks 3 and 4: the surplus closes ---------------------------------------------------
close_P = MS_P - (B.rent_P.sum() + shunt_rent_P)
close_Q = MS_Q - (B.rent_Q.sum() + shunt_rent_Q)

cols = ['branch', 'i', 'k', 'Pik_MW', 'Pki_MW', 'loss_P', 'loss_rent_P', 'spread_rent_P',
        'rent_P', 'mu_Sik', 'mu_Ski', 'Loading', 'binding']
print('TIER 2 - active-power rent, branch by branch\n')
print(B[cols].to_string(index=False, float_format=lambda x: f'{x:12.6f}'))
print(f"\n  loss rent        {B.loss_rent_P.sum():16.6f}   (mean price x loss)")
print(f"  spread rent      {B.spread_rent_P.sum():16.6f}   (price spread x mean directed flow)")
print(f"  shunt rent       {shunt_rent_P:16.6f}")
print(f"  ---------------------------------")
print(f"  sum              {B.rent_P.sum() + shunt_rent_P:16.6f}")
print(f"  MS_P (Tier 1)    {MS_P:16.6f}")
print(f"  RESIDUAL         {close_P:16.3e}   <- must be solver tolerance, not economics")
print(f"\n  same, directed split:  loss {B.loss_rent_dir_P.sum():.6f}  "
      f"congestion {B.spread_rent_dir_P.sum():.6f}")
print(f"\n  reactive:  loss {B.loss_rent_Q.sum():.6f}   spread {B.spread_rent_Q.sum():.6f}   "
      f"shunt {shunt_rent_Q:.6f}   MS_Q {MS_Q:.6f}   residual {close_Q:.3e}")

assert abs(close_P) < TOL, 'the active-power surplus does not close on the branch decomposition'
assert abs(close_Q) < TOL, 'the reactive-power surplus does not close on the branch decomposition'
print('\n  -> both close. The surplus is exactly the network rent, nothing else.')
if not B.binding.any():
    print('  -> no branch limit binds: every euro above is loss-driven.')


In [ ]:
# --- independent cross-check: rebuild Ybus from the archived inputs -----------------------
# The closure above used only the exported flow columns. This rebuilds the network from
# Inputs/line_data.csv and recomputes the injection in polar form, so a bad flow column
# cannot hide inside a consistent-looking identity.
#
#   P_k = V_k * sum_m V_m (G_km cos(th_km) + B_km sin(th_km)),  th_km = th_k - th_m
#
# (The trig pair is G*cos + B*sin, not G*sin + B*cos.) Bus shunts are deliberately NOT put
# into Y: the balance row keeps g_sh*V^2 as its own term, so P_MW excludes it.
LINES = rd_opt('Inputs', 'line_data.csv')
YBUS_OK = LINES is not None

if YBUS_OK:
    ratio = LINES['ratio'].replace(0.0, 1.0) if 'ratio' in LINES else pd.Series(1.0, index=LINES.index)
    angle = LINES['angle'] if 'angle' in LINES else pd.Series(0.0, index=LINES.index)
    if ((ratio - 1.0).abs() > 1e-12).any() or (angle.abs() > 1e-12).any():
        print('skipped: this case has off-nominal taps or phase shifters, and the Ybus built')
        print('         here models neither. The Tier 2 closure above is unaffected.')
        YBUS_OK = False

if YBUS_OK:
    ids = list(BUS.index)
    pos = {b: j for j, b in enumerate(ids)}
    n   = len(ids)
    Gm  = np.zeros((n, n))
    Bm  = np.zeros((n, n))
    for _, r in LINES.iterrows():
        if 'status' in LINES and float(r['status']) == 0.0:
            continue
        a, b_ = pos[int(r['fbus'])], pos[int(r['tbus'])]
        y = 1.0 / complex(float(r['r']), float(r['x']))
        g, s = y.real, y.imag
        half = float(r['b']) / 2.0
        Gm[a, a] += g;  Bm[a, a] += s + half
        Gm[b_, b_] += g; Bm[b_, b_] += s + half
        Gm[a, b_] -= g; Bm[a, b_] -= s
        Gm[b_, a] -= g; Bm[b_, a] -= s

    Vm  = BUS.V_pu.to_numpy(float)
    th  = np.radians(BUS.Theta_deg.to_numpy(float))
    dth = th[:, None] - th[None, :]
    Pin = (Vm[:, None] * Vm[None, :] * (Gm * np.cos(dth) + Bm * np.sin(dth))).sum(axis=1) * SBASE
    Qin = (Vm[:, None] * Vm[None, :] * (Gm * np.sin(dth) - Bm * np.cos(dth))).sum(axis=1) * SBASE

    dP = np.abs(Pin - BUS.P_MW.to_numpy(float)).max()
    dQ = np.abs(Qin - BUS.Q_MVAr.to_numpy(float)).max()
    print(f'max |P_ybus - P_report|    {dP:.3e} MW')
    print(f'max |Q_ybus - Q_report|    {dQ:.3e} MVAr')

    ms_from_ybus_P = float(-(lam.to_numpy(float) * Pin).sum()) + shunt_rent_P
    ms_from_ybus_Q = float(-(muQ.to_numpy(float) * Qin).sum()) + shunt_rent_Q
    print(f'\nMS_P from Ybus injection   {ms_from_ybus_P:16.6f}   vs Tier 1 {MS_P:16.6f}'
          f'   gap {ms_from_ybus_P - MS_P:.3e}')
    print(f'MS_Q from Ybus injection   {ms_from_ybus_Q:16.6f}   vs Tier 1 {MS_Q:16.6f}'
          f'   gap {ms_from_ybus_Q - MS_Q:.3e}')
    if dP > 1e-6 or dQ > 1e-6:
        print('\nWARNING: the rebuilt injection disagrees with buses_report. Advisory only -')
        print('         this cell models no taps, no phase shift and no bus shunt in Y.')
else:
    ms_from_ybus_P = float('nan')
    ms_from_ybus_Q = float('nan')


In [ ]:
# --- generator side: cost recovery and the stationarity identity ---------------------------
# Cost curve from the manifest [resolved] block, so nothing is hardcoded per case.
g_ids = [int(x) for x in arr('gen_ids')]
g_bus = [int(x) for x in arr('gen_bus')]
COST  = pd.DataFrame({'gen': g_ids, 'c0': arr('c0'), 'c1': arr('c1'), 'c2': arr('c2')}).set_index('gen')

G = GENR.rename(columns={'ID': 'gen', 'BUS': 'bus'})[['gen', 'bus', 'P_MW', 'Q_MVAr']].copy()
G = G.join(COST, on='gen')
G['lam']       = G.bus.map(lam)
G['cost']      = G.c0 + G.c1 * G.P_MW + G.c2 * G.P_MW ** 2
G['marg_cost'] = G.c1 + 2.0 * G.c2 * G.P_MW
G['mu_Pg_lower'] = G.gen.map(pg_lo)
G['mu_Pg_upper'] = G.gen.map(pg_up)
G['mu_Qg_lower'] = G.gen.map(qg_lo)
G['mu_Qg_upper'] = G.gen.map(qg_up)
G['revenue'] = G.lam * G.P_MW
G['profit']  = G.revenue - G.cost

# Stationarity, with the senses fixed by bound_builders.jl:160,169:
#   lower row   lb - P_g <= 0    d/dP_g = -1
#   upper row   P_g - ub <= 0    d/dP_g = +1
#   balance     P_g - ... == 0   d/dP_g = +1
# contribution = -dual * coeff and sum(contributions) + C' = 0, hence
#   lambda = C' + mu_lower - mu_upper.
G['stationarity'] = G.marg_cost - G.lam + G.mu_Pg_lower - G.mu_Pg_upper
if SG_UPPER and G.gen.map(sg_up).abs().max() > 1e-9:
    print('NOTE: |S_g| capability duals are nonzero, and this column does not carry their')
    print('      quadratic term. Read kkt_Pg_terms.csv for the complete decomposition.')

uplift = float(np.maximum(0.0, -G.profit).sum())

print('GENERATOR SIDE\n')
show = ['gen', 'bus', 'P_MW', 'lam', 'marg_cost', 'mu_Pg_lower', 'mu_Pg_upper',
        'revenue', 'cost', 'profit', 'stationarity']
print(G[show].to_string(index=False, float_format=lambda x: f'{x:12.6f}'))
print(f'\n  total revenue {G.revenue.sum():14.6f}   total cost {G.cost.sum():14.6f}   '
      f'total profit {G.profit.sum():14.6f}')
print(f'  make-whole owed at lambda  {uplift:14.6f} EUR/h')
print(f'  objective from manifest    {OBJ_MVA:14.6f} EUR/h   '
      f'gap vs total cost {G.cost.sum() - OBJ_MVA:.3e}')

# --- check 6: against the Julia-side KKT export, when the run carries it -------------------
KT = rd_opt('Dispatch', 'CSV_duals', 'kkt_Pg_terms.csv')
if KT is not None:
    piv = KT.pivot_table(index='gen_id', columns='family', values='contribution',
                         aggfunc='sum').fillna(0.0)
    # A unit with status = 0 never reaches the model, so the export can legitimately hold
    # fewer generators than generators_report. Compare on the intersection and say so,
    # rather than reindexing a missing row into a NaN that fails as an unreadable assert.
    common = [g for g in G.gen.values if g in piv.index]
    if len(common) != len(G):
        print(f'  note: {len(G) - len(common)} generator(s) in the report are absent from '
              'kkt_Pg_terms.csv (inactive units); comparing the rest.')
    Gk = G.set_index('gen').loc[common]
    resid = piv['RESIDUAL'].reindex(common).to_numpy(float)
    print(f'\nkkt_Pg_terms.csv present: max |RESIDUAL| {np.abs(resid).max():.3e}')
    assert np.abs(resid).max() < 1e-8, 'the exported P_g stationarity does not close'
    lam_kkt = (-piv['p_balance']).reindex(common).to_numpy(float)   # contribution = -lambda*1
    gap_lam = np.abs(lam_kkt - Gk.lam.to_numpy(float)).max()
    cg      = piv['cost_gradient'].reindex(common).to_numpy(float)
    gap_cg  = np.abs(cg - Gk.marg_cost.to_numpy(float)).max()
    print(f'  max |lambda_kkt - lambda_dual|   {gap_lam:.3e} EUR/MW')
    print(f'  max |C\'_kkt - C\'_manifest|       {gap_cg:.3e} EUR/MW')
    assert gap_lam < TOL, 'kkt_Pg_terms lambda disagrees with dual_P_balance'
    assert gap_cg  < TOL, 'kkt_Pg_terms cost gradient disagrees with the manifest cost curve'
    print('  -> the Julia export and this notebook agree on every generator.')
else:
    print('\nno kkt_Pg_terms.csv: set post_solve_hook in the run script for the Julia-side check.')


In [ ]:
# --- the verdict ---------------------------------------------------------------------------
loss_P, spread_P = float(B.loss_rent_P.sum()), float(B.spread_rent_P.sum())
loss_Q, spread_Q = float(B.loss_rent_Q.sum()), float(B.spread_rent_Q.sum())

VER = pd.DataFrame({
    'quantity': [
        'load pays (active)', 'load pays (reactive)',
        'generation paid (active)', 'generation paid (reactive)',
        'OPERATOR SURPLUS (active)', '  of which loss rent', '  of which spread rent',
        '  of which shunt rent', '  closure residual',
        'OPERATOR SURPLUS (reactive)', '  of which loss rent', '  of which spread rent',
        '  of which shunt rent', '  closure residual',
        'generator revenue', 'generator cost', 'generator profit', 'make-whole owed',
        'objective (manifest)', 'cost recovery residual',
    ],
    'EUR_per_h': [
        pay_load_P, pay_load_Q, pay_gen_P, pay_gen_Q,
        MS_P, loss_P, spread_P, shunt_rent_P, close_P,
        MS_Q, loss_Q, spread_Q, shunt_rent_Q, close_Q,
        float(G.revenue.sum()), float(G.cost.sum()), float(G.profit.sum()), uplift,
        OBJ_MVA, float(G.revenue.sum() - G.cost.sum() - G.profit.sum()),
    ],
})
print('DOES THE MONEY CLOSE?\n')
print(VER.to_string(index=False, float_format=lambda x: f'{x:18.8f}'))
print()
if abs(close_P) < TOL and abs(close_Q) < TOL:
    print(f'YES. Both surpluses reproduce the network rent to {max(abs(close_P), abs(close_Q)):.1e} '
          'EUR/h,')
    print('which is the solver tolerance, not a modelling gap. Every euro the operator holds')
    print('is accounted for by losses, by the price spread across branches, and by shunts.')
    print('No voltage or generator-bound dual contributes to it - those pay for themselves')
    print('inside the generator profit column instead.')
else:
    print('NO. See the residual rows above.')


In [ ]:
# --- figures --------------------------------------------------------------------------------
# Displayed only. nbconvert embeds them in the HTML report, so writing standalone files
# beside it would duplicate the same picture into the run folder - the KKT notebooks never
# did, and this matches them.
plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'dejavuserif',
    'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'grid.linewidth': 0.35, 'grid.color': '0.75',
    'legend.framealpha': 0.85, 'savefig.bbox': 'tight',
    'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
BLUE, ORANGE, GREEN, RED, GREY = '#0072B2', '#D55E00', '#009E73', '#c44e52', '#8c8c8c'

# (a) waterfall: payments into the surplus, then the surplus into its parts.
# Two groups, each cumulative from zero, each closing on the same number.
labels = ['load\npays', '- gen\npaid', 'OPERATOR\nSURPLUS',
          'loss\nrent', 'spread\nrent', 'shunt\nrent', 'residual', 'SUM OF\nRENTS']
vals   = [pay_load_P, -pay_gen_P, None, loss_P, spread_P, shunt_rent_P, close_P, None]
totals = {2: MS_P, 7: loss_P + spread_P + shunt_rent_P + close_P}

fig, ax = plt.subplots(figsize=(9.5, 4.2))
run_tot = 0.0
for j, (lab, d) in enumerate(zip(labels, vals)):
    if d is None:
        t = totals[j]
        ax.bar(j, t, color=BLUE)
        ax.text(j, t, f'{t:,.2f}', ha='center', va='bottom' if t >= 0 else 'top',
                fontsize=8, fontweight='bold')
        run_tot = 0.0                       # the next group restarts from zero
    else:
        ax.bar(j, d, bottom=run_tot, color=GREEN if d >= 0 else RED)
        ax.text(j, run_tot + d, f'{d:+,.2f}', ha='center',
                va='bottom' if d >= 0 else 'top', fontsize=7)
        run_tot += d
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(2.5, color='0.6', lw=0.9, ls=':')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_ylabel('EUR/h')
ax.set_title(f'Active-power settlement and the network rent that must equal it'
             f'  -  {CASE}  -  {RUN.name}')
lo, hi = ax.get_ylim()
ax.text(1.0, hi - 0.06 * (hi - lo), 'settlement', ha='center', fontsize=8, color='0.4')
ax.text(5.0, hi - 0.06 * (hi - lo), 'network rent', ha='center', fontsize=8, color='0.4')
fig.tight_layout(); plt.show()

# (b) nodal prices with the demand that pays them
xb = np.arange(len(BUS))
fig2, ax2 = plt.subplots(figsize=(9.5, 3.4))
ax2.bar(xb, lam.to_numpy(float), color=BLUE, width=0.6, label=r'$\lambda_k$')
ax2.set_ylabel('EUR/MW'); ax2.set_xticks(xb)
ax2.set_xticklabels([f'{b}\n{p:.0f} MW' if p > 0 else f'{b}\n-' for b, p in
                     zip(BUS.index, BUS.PD_MW)])
ax2.set_xlabel('bus / demand')
ax2.set_title('Nodal active prices')
ax2.legend(); fig2.tight_layout(); plt.show()

# (c) branch rent, split, with binding branches marked
xr = np.arange(len(B))
fig3, ax3 = plt.subplots(figsize=(9.5, 3.6))
ax3.bar(xr - 0.2, B.loss_rent_P,   width=0.4, color=ORANGE, label='loss rent')
ax3.bar(xr + 0.2, B.spread_rent_P, width=0.4, color=GREEN,  label='spread rent')
for j, bind in enumerate(B.binding):
    if bind:
        ax3.annotate('binding', (j, 0), textcoords='offset points', xytext=(0, 6),
                     ha='center', fontsize=7, color=RED)
ax3.axhline(0, color='gray', lw=0.8)
ax3.set_xticks(xr)
ax3.set_xticklabels([f'{int(i)}-{int(k)}' for i, k in zip(B.i, B.k)])
ax3.set_xlabel('branch'); ax3.set_ylabel('EUR/h')
ax3.set_title('Where the network rent is collected')
ax3.legend(); fig3.tight_layout(); plt.show()

# (d) generator prices against marginal cost
xg = np.arange(len(G))
fig4, ax4 = plt.subplots(figsize=(9.5, 3.6))
ax4.bar(xg - 0.2, G.lam,       width=0.4, color=BLUE, label=r'$\lambda_g$')
ax4.bar(xg + 0.2, G.marg_cost, width=0.4, color=GREY, label=r"$C'_g$")
ax4b = ax4.twinx()
ax4b.plot(xg, G.profit, 'ko-', ms=5, lw=1.2, label='profit [EUR/h]')
ax4b.axhline(0, color='gray', lw=0.6)
ax4b.set_ylabel('EUR/h')
ax4.set_xticks(xg)
ax4.set_xticklabels([f'G{int(g)}\nbus {int(b)}\n{p:.1f} MW'
                     for g, b, p in zip(G.gen, G.bus, G.P_MW)])
ax4.set_ylabel('EUR/MW')
ax4.set_title(r'Generator price vs marginal cost; the gap IS the bound rent')
h1, l1 = ax4.get_legend_handles_labels(); h2, l2 = ax4b.get_legend_handles_labels()
ax4.legend(h1 + h2, l1 + l2, loc='upper left')
fig4.tight_layout(); plt.show()


In [ ]:
# --- write the tables beside the data they describe -----------------------------------------
BUSOUT = pd.DataFrame({
    'bus': BUS.index,
    'PD_MW': BUS.PD_MW.to_numpy(float), 'QD_MVAr': BUS.QD_MVAr.to_numpy(float),
    'PG_MW': BUS.PG_MW.to_numpy(float), 'QG_MVAr': BUS.QG_MVAr.to_numpy(float),
    'Psh_MW': BUS.Psh_MW.to_numpy(float), 'Qsh_eff_MVAr': Qsh_eff.to_numpy(float),
    'V_pu': BUS.V_pu.to_numpy(float), 'theta_deg': BUS.Theta_deg.to_numpy(float),
    'lambda': lam.to_numpy(float), 'mu_Q': muQ.to_numpy(float),
    'sigma_pin': sigma.to_numpy(float),
    'mu_V_lower': v_lo.to_numpy(float), 'mu_V_upper': v_up.to_numpy(float),
    'pay_load_P': (lam * BUS.PD_MW).to_numpy(float),
    'pay_gen_P':  (lam * BUS.PG_MW).to_numpy(float),
    'pay_load_Q': (muQ * BUS.QD_MVAr).to_numpy(float),
    'pay_gen_Q':  (muQ * BUS.QG_MVAr).to_numpy(float),
})

BROUT = B[['branch', 'i', 'k', 'Pik_MW', 'Pki_MW', 'Qik_MVAr', 'Qki_MVAr',
           'loss_P', 'loss_rent_P', 'spread_rent_P', 'loss_rent_dir_P',
           'spread_rent_dir_P', 'rent_P', 'loss_Q', 'loss_rent_Q', 'spread_rent_Q',
           'rent_Q', 'mu_Sik', 'mu_Ski', 'Sik_MVA', 'Ski_MVA', 'Cap_MVA',
           'Loading', 'binding']].copy()

GOUT = G[['gen', 'bus', 'P_MW', 'Q_MVAr', 'c0', 'c1', 'c2', 'lam', 'marg_cost',
          'mu_Pg_lower', 'mu_Pg_upper', 'mu_Qg_lower', 'mu_Qg_upper',
          'revenue', 'cost', 'profit', 'stationarity']].rename(columns={'lam': 'lambda'})

SUM = VER.copy()
SUM = pd.concat([SUM, pd.DataFrame({
    'quantity': ['MS_P_from_ybus', 'MS_Q_from_ybus', 'total_demand_MW',
                 'total_generation_MW', 'losses_MW', 'lambda_min', 'lambda_max',
                 'max_abs_sigma_minus_lambda', 'n_binding_branches'],
    'EUR_per_h': [ms_from_ybus_P, ms_from_ybus_Q, float(BUS.PD_MW.sum()),
                  float(BUS.PG_MW.sum()),
                  float(BUS.PG_MW.sum() - BUS.PD_MW.sum() - BUS.Psh_MW.sum()),
                  float(lam.min()), float(lam.max()),
                  SIGMA_GAP,
                  float(B.binding.sum())],
})], ignore_index=True)

BUSOUT.to_csv(RUN / 'acopf_settlement_bus.csv',     sep=';', index=False)
BROUT.to_csv(RUN / 'acopf_settlement_branch.csv',   sep=';', index=False)
GOUT.to_csv(RUN / 'acopf_settlement_gen.csv',       sep=';', index=False)
SUM.to_csv(RUN / 'acopf_settlement_summary.csv',    sep=';', index=False)
for f in ['acopf_settlement_bus.csv', 'acopf_settlement_branch.csv',
          'acopf_settlement_gen.csv', 'acopf_settlement_summary.csv']:
    print(f'written: {RUN / f}')


## Reading the numbers

**The residual rows are the headline.** `closure residual` on active and reactive
power must sit at the solver's dual tolerance — around `1e-9` at `tol = 1e-8`.
Anything larger is not a market finding, it is a broken read: a flow column with
the wrong sign, a shunt counted twice, a dual file from a different run. The whole
point of running this on plain ACOPF is that there is no legitimate reason for it
to be nonzero.

**A zero spread rent is a valid answer.** It means no thermal limit binds, so the
only thing separating one bus price from another is losses. Check
`n_binding_branches` in the summary before reading anything into a small number.
If it is zero and the run was made with `ineq_sbranch_upper = false`, congestion
was never modelled at all.

**The two splits disagree on labels, not on money.** `loss_rent_P` uses the mean of
the two end prices, `loss_rent_dir_P` uses the to-side price; their partners absorb
the difference. Both sum to `rent_P`. Quote the symmetric pair unless you need the
textbook congestion-rent expression, because the directed one changes sign if the
branch is entered from the other end.

**Sign of the surplus is not a bug.** Losses push it down — more is generated than
consumed, and every MW of that difference is paid for. The loss-driven price spread
pushes it up. On a lossy AC network with marginal loss pricing the spread term
normally wins, but nothing in the identity forces it to.

**$\mu_V$ and $\mu_\theta$ are exported and are not in the surplus.** They appear in
`acopf_settlement_bus.csv` for completeness only. A binding voltage limit changes
the dispatch and therefore changes $\lambda$, but it collects no rent of its own —
the network identity has no term for it. Anyone adding a voltage charge on top of
LMP settlement would be collecting the same money twice.

**A make-whole under linear cost means a unit is pinned.** With $C'$ constant, a
generator strictly inside its bounds earns exactly zero profit: $\lambda = C'$. A
nonzero `make-whole owed` therefore says some unit is held at $P_{\min}$ against
the merit order, and $\lambda < C'$ there. That is the ACOPF baseline for the
uplift the transient-stability case reports — the difference between the two is
what the stability constraint costs, and only the difference.

**$\sigma = \lambda$ is the load-bearing check.** It is trivially true here, which
is the point. The same notebook run against a TSC case would show
$\sigma - \lambda \neq 0$, and having proved the gap is zero without stability rows
is what makes that difference attributable.
